# FADAS: adaptive server optimizer (K=5)

Buffered async FL in which the server updates the global model with an **Adam-style optimizer** (AMSGrad) instead of a plain step. Based on Wang et al., *FADAS: Towards Federated Adaptive Asynchronous Optimization* (ICML 2024).

**How it works**

1. Clients train continuously; arriving updates wait in a buffer, as in FedBuff.
2. When the buffer is full, the server averages the buffered deltas: `delta = mean(delta_i)`.
3. It treats `delta` as a gradient and updates its optimizer state: `m = beta1 * m + (1 - beta1) * delta`, `v = beta2 * v + (1 - beta2) * delta^2`, `v_hat = max(v_hat, v)`.
4. New global model: `global += rate * m / (sqrt(v_hat) + eps)`.
5. With `delay_adaptive = True` (FADASda): if the stalest update in the buffer is more than `delay_threshold` versions old, the step uses `rate / max_delay` instead.

> **Note:** the adaptive step gives every weight its own step size. Weights whose updates keep changing sign get small steps, and the momentum `m` smooths out the noise of stale and non-IID updates.

> **Important:** `server_rate` means something different here than in FedBuff. Each step moves every weight by roughly `server_rate`, so it is small (0.003 here; the paper uses 0.0001 to 0.001 over 500 rounds). It was chosen on validation accuracy: 0.001 and 0.003 tie (88.0%), 0.01 drops to 74%.

> **Caution:** simplifications compared with the paper:
> * The delay-adaptive step is `rate / max_delay`, as in the paper's implementation details (Appendix D.2); the main text writes `min(rate, 1 / max_delay)`.
> * With the shared settings no update is more than about 5 versions stale, so `delay_adaptive = True` gives the same result as False. It matters with real stragglers: see `paper_fadas_vs_fedbuff.ipynb`.
> * The paper tunes the local learning rate per method (0.1 for FADAS); here every approach uses the shared `lr` from `settings.yaml`.
> * By default **all** clients train at once, like the other approaches. Set `concurrency` to use the paper's random sampling of `M_c` active clients.

## 1. Load the simulator

Gives this notebook everything from `simulator.ipynb`: the settings, `Approach`, `fedavg`, `apply_delta` and `run`.

In [1]:
%run ../simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class FADAS(Approach):
    name = "fadas"
    label = "FADAS adaptive K=5"          # shown in plot legends
    buffer_size = 5               # updates per global step (M in the paper)
    server_rate = 0.003           # global step size (eta); with Adam each weight moves by about this much per step
    beta1 = 0.9                   # momentum of the averaged update
    beta2 = 0.99                  # decay of the squared-update average
    eps = 1e-8
    delay_adaptive = False        # True = FADASda: smaller step when the buffer holds very stale updates
    delay_threshold = 8           # tau_c: delays up to this keep the full step
    concurrency = None            # clients training at once (M_c); None = all clients, like the other approaches

    def start(self, server):
        zeros = {k: torch.zeros_like(v) for k, v in server.model.items()}
        self.m, self.v, self.v_hat = dict(zeros), dict(zeros), dict(zeros)
        self.buffer = []
        if self.concurrency is None:
            super().start(server)
        else:
            for cid in server.rng.choice(len(server.clients), self.concurrency, replace=False):
                server.train(int(cid))

    def client_free(self, server, client):
        if self.concurrency is None:
            server.train(client.id)
        else:                                              # sample another idle client
            idle = [c.id for c in server.clients if not c.busy]
            server.train(int(server.rng.choice(idle)))

    def on_update(self, server, update):
        self.buffer.append(update)
        if len(self.buffer) < self.buffer_size:
            return
        rate = self.server_rate
        max_delay = max(u.staleness for u in self.buffer)
        if self.delay_adaptive and max_delay > self.delay_threshold:
            rate = self.server_rate / max_delay
        new_model = {}
        for k, cur in server.model.items():
            if not torch.is_floating_point(cur):
                new_model[k] = cur.clone()
                continue
            delta = sum(u.delta[k] for u in self.buffer) / len(self.buffer)
            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * delta
            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * delta * delta
            self.v_hat[k] = torch.maximum(self.v_hat[k], self.v[k])
            new_model[k] = cur + rate * self.m[k] / (self.v_hat[k].sqrt() + self.eps)
        server.set_model(new_model, self.buffer)
        self.buffer = []

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(FADAS(delay_adaptive=True))` for FADASda, or `run(FADAS(server_rate=0.001))`. It is saved as a separate run.

In [3]:
results = run(FADAS())

fadas                        seed=42   val= 89.14%  acc= 90.40%  time=   560.2s  updates=  100  worst10%= 87.81%  wall=8.9s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
20,42,560.1644,100,0.904,0.3142,0.8781,2.8
